# Variable Phase Shift + Gain — Red Pitaya control

Controls the custom `red_pitaya_phaseshift` FPGA module (Herkenhoff IQ phase shifter)
on system-bus slot **`sys[3]` → base `0x40300000`** (the PID's old slot).

**Signal path:** signal generator → ADC CH A → phase/gain module → DAC CH A → oscilloscope.

**How this notebook talks to the board:** the FPGA registers live in the ARM's physical
address space at `0x40300000`. This notebook maps that page via `/dev/mem` and writes 4-byte
words to it; each write is an AXI bus write straight into a hardware register. The FPGA runs
the ADC→module→DAC path continuously at 125 MHz on its own — Python only sets the knobs.

*Compatible with the board's Python 3.5 kernel (no f-strings). Must run as root.*

### Register map (offsets from `0x40300000`)

| Offset | Name | Format | Meaning |
|--------|------|--------|---------|
| 0x00 | CI    | signed Q2.14   | gain·cos(φ) |
| 0x04 | CQ    | signed Q2.14   | gain·sin(φ) |
| 0x08 | DLY   | uint (0..1022) | ⌊D⌋ integer delay, D = f_clk/(4·f_sig) |
| 0x0C | DELTA | uint Q0.14     | δ = D − ⌊D⌋ |
| 0x10 | BYP   | bit            | 1 = bypass (pass input through) |

The module has a fixed **2-sample** pipeline latency; `configure()` compensates automatically
so the scope phase matches what you request.

In [ ]:
import os, mmap, struct
import numpy as np

FS    = 125e6          # FPGA / ADC / DAC sample clock [Hz]
FRAC  = 14
SCALE = 1 << FRAC      # 16384 -> fixed-point 1.0
BASE  = 0x40300000     # sys[3] region (was PID)
MAP   = 0x1000         # one 4 KB page
DEPTH = 1024           # delay-line length in the HDL
L_PIPE= 3              # fixed module latency in samples (BRAM read + mult + FIX 6 output pipeline)

REG = {'CI':0x00, 'CQ':0x04, 'DLY':0x08, 'DELTA':0x0C, 'BYP':0x10}

class PhaseShifter(object):
    def __init__(self, base=BASE):
        self.fd  = os.open('/dev/mem', os.O_RDWR | os.O_SYNC)
        self.mem = mmap.mmap(self.fd, MAP, offset=base)

    # struct.pack/unpack works on any Python 2.7 / 3.x (no int.to_bytes needed)
    def _wr(self, off, val):
        self.mem[off:off+4] = struct.pack('<I', val & 0xFFFFFFFF)
    def _rd(self, off):
        return struct.unpack('<I', self.mem[off:off+4])[0]

    @staticmethod
    def _s16(x):
        x = int(round(x)); x = max(-32768, min(32767, x)); return x & 0xFFFF

    def configure(self, phase_deg, gain, signal_freq):
        # cancel the module's fixed 3-sample transport latency so the phase on
        # the scope matches the phase you asked for
        phi_eff = phase_deg - 360.0*L_PIPE*signal_freq/FS
        phi     = np.deg2rad(phi_eff)
        ci      = self._s16(gain*np.cos(phi)*SCALE)
        cq      = self._s16(gain*np.sin(phi)*SCALE)
        D       = FS/(4.0*signal_freq)
        dly     = min(int(np.floor(D)), DEPTH-2)
        delta   = min(int(round((D-np.floor(D))*SCALE)), SCALE-1)
        self._wr(REG['CI'],    ci)
        self._wr(REG['CQ'],    cq)
        self._wr(REG['DLY'],   dly)
        self._wr(REG['DELTA'], delta)
        self._wr(REG['BYP'],   0)
        if D > DEPTH-2:
            fmin = FS/4/(DEPTH-2)/1e3
            print('WARNING: D={0:.1f} exceeds delay depth; raise DEPTH in HDL for f<{1:.0f} kHz'.format(D, fmin))
        return dict(ci=ci, cq=cq, dly=dly, delta=delta, D=round(D,3), phi_eff=round(phi_eff,2))

    def bypass(self, on=True):
        self._wr(REG['BYP'], 1 if on else 0)

    def readback(self):
        raw = dict((k, self._rd(o)) for k, o in REG.items())
        for k in ('CI','CQ'):                       # show the signed coefficients
            if raw[k] & 0x8000:
                raw[k] -= 0x10000
        return raw

    def close(self):
        self.mem.close(); os.close(self.fd)

ps = PhaseShifter()
print('Phase shifter mapped at 0x%08X' % BASE)

## Interactive control
Move the sliders — every change writes the registers immediately.
Set **Signal freq** to match your signal-generator frequency.

If the sliders do not appear (ipywidgets missing on the board), skip this cell and use the
manual cell below.

In [ ]:
import ipywidgets as widgets

def _apply(phase_deg, gain, signal_freq_kHz):
    info = ps.configure(phase_deg, gain, signal_freq_kHz*1e3)
    print('phi={0:+6.1f} deg   gain={1:4.2f}   f={2:6.1f} kHz'.format(phase_deg, gain, signal_freq_kHz))
    print('  -> CI={0}  CQ={1}  DLY={2}  DELTA={3}  (D={4}, phi_eff={5})'.format(
          info['ci'], info['cq'], info['dly'], info['delta'], info['D'], info['phi_eff']))

widgets.interact(
    _apply,
    phase_deg       = widgets.FloatSlider(value=0,   min=-180, max=180, step=1,   description='Phase [deg]'),
    gain            = widgets.FloatSlider(value=1.0, min=0.0,  max=1.0, step=0.01, description='Gain'),
    signal_freq_kHz = widgets.FloatSlider(value=740, min=30,   max=2000,step=1,   description='Signal freq [kHz]'),
);

## Manual set / read (no widgets)

In [ ]:
# one-shot configuration
info = ps.configure(phase_deg=45, gain=1.0, signal_freq=740e3)
print('wrote:', info)
print('readback:', ps.readback())

# ps.bypass(True)    # uncomment to send the input straight through (sanity check)
# ps.bypass(False)

## Register smoke test (optional)
Write a known value and read it back to prove the `sys[3]` bus is alive before wiring the scope.

In [ ]:
ps._wr(REG['CI'], 0x4000)                       # CI = 16384 -> gain*cos = 1.0
print('CI readback: 0x%04X (expect 0x4000)' % (ps._rd(REG['CI']) & 0xFFFF))
ps.configure(phase_deg=0, gain=1.0, signal_freq=740e3)   # restore a sane state

# When completely finished:
# ps.close()